In [3]:
import time
import requests
import pandas as pd

from datetime import datetime

HEADERS = {"User-Agent": "Research Script (contact: tz25@fsu.edu)"}

def cik_pad(cik: str) -> str:
    s = str(cik).strip()
    if not s.isdigit():
        raise ValueError(f"Bad CIK: {cik}")
    return str(int(s)).zfill(10)

def accession_nodash(acc: str) -> str:
    return acc.replace("-", "")

def archives_base(cik10: str, acc: str) -> str:
    return f"https://www.sec.gov/Archives/edgar/data/{int(cik10)}/{accession_nodash(acc)}/"

def fetch_submissions_json(cik: str) -> dict:
    cik10 = cik_pad(cik)
    url = f"https://data.sec.gov/submissions/CIK{cik10}.json"
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.json()

def extract_10k_rows(block: dict, cik10: str, start_year: int, end_year: int, include_amended: bool):
    forms = block.get("form", [])
    filing_dates = block.get("filingDate", [])
    report_dates = block.get("reportDate", [])
    accessions = block.get("accessionNumber", [])
    primary_docs = block.get("primaryDocument", [])

    out = []
    for form, fdate, rdate, acc, pdoc in zip(forms, filing_dates, report_dates, accessions, primary_docs):
        # form filter
        if include_amended:
            if form not in ("10-K", "10-K/A"):
                continue
        else:
            if form != "10-K":
                continue

        # use reportDate year as fiscal year filter (preferred for 10-K)
        try:
            fy = datetime.strptime(rdate, "%Y-%m-%d").year
        except Exception:
            # if reportDate missing/bad, you can fall back to filingDate year
            try:
                fy = datetime.strptime(fdate, "%Y-%m-%d").year
            except Exception:
                continue

        if fy < start_year or fy > end_year:
            continue

        base = archives_base(cik10, acc)
        out.append({
            "CIK": cik10,
            "Company": None,          # fill later
            "filingDate": fdate,
            "reportDate": rdate,
            "fiscalYear": fy,
            "accessionNumber": acc,
            "html": base + pdoc,
        })
    return out

def get_10k_html_urls_from_submissions(cik: str, start_year=2015, end_year=2025, include_amended=False, sleep_sec=0.2):
    data = fetch_submissions_json(cik)
    cik10 = cik_pad(cik)

    rows = []
    recent = data.get("filings", {}).get("recent", {})
    rows.extend(extract_10k_rows(recent, cik10, start_year, end_year, include_amended))

    for f in data.get("filings", {}).get("files", []):
        name = f.get("name")
        if not name:
            continue
        chunk_url = "https://data.sec.gov/submissions/" + name
        r = requests.get(chunk_url, headers=HEADERS, timeout=30)
        r.raise_for_status()
        chunk = r.json()
        rows.extend(extract_10k_rows(chunk, cik10, start_year, end_year, include_amended))
        time.sleep(sleep_sec)

    df = (pd.DataFrame(rows)
          .drop_duplicates(subset=["accessionNumber"])
          .sort_values(["CIK", "fiscalYear", "filingDate"])
          .reset_index(drop=True))

    df["Company"] = data.get("name")
    return df[["Company", "CIK", "fiscalYear", "html"]]

def export_company_cik_year_html(cik_ls,
                                 out_xlsx="company_2015_2025.xlsx",
                                 start_year=2015,
                                 end_year=2025,
                                 include_amended=False,
                                 sleep_sec=0.2):
    if isinstance(cik_ls, str):
        raise TypeError("cik_ls must be a LIST, e.g. ['0001318605','0000320193']")

    all_dfs = []
    errors = []

    for i, cik in enumerate(cik_ls, 1):
        try:
            print(f"[{i}/{len(cik_ls)}] CIK={cik}")
            df = get_10k_html_urls_from_submissions(
                cik,
                start_year=start_year,
                end_year=end_year,
                include_amended=include_amended,
                sleep_sec=sleep_sec
            )
            all_dfs.append(df)
        except Exception as e:
            errors.append({"CIK": str(cik), "error": str(e)})
            print(f"  Failed: {e}")

    df_all = pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame(
        columns=["Company", "CIK", "fiscalYear", "html"]
    )

    # save to Excel (better than CSV for your use case)
    with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
        df_all.to_excel(writer, index=False, sheet_name="10K_HTML")
        if errors:
            pd.DataFrame(errors).to_excel(writer, index=False, sheet_name="Errors")

    print(f"Saved: {out_xlsx}")
    return df_all



cik_ls = ["0001341439", "0001108524", "0001318605", "0000012927"]

df_final = export_company_cik_year_html(cik_ls)
print(df_final.head(10))

[1/4] CIK=0001341439
[2/4] CIK=0001108524
[3/4] CIK=0001318605
[4/4] CIK=0000012927
Saved: company_2015_2025.xlsx
       Company         CIK  fiscalYear  \
0  ORACLE CORP  0001341439        2015   
1  ORACLE CORP  0001341439        2016   
2  ORACLE CORP  0001341439        2017   
3  ORACLE CORP  0001341439        2018   
4  ORACLE CORP  0001341439        2019   
5  ORACLE CORP  0001341439        2020   
6  ORACLE CORP  0001341439        2021   
7  ORACLE CORP  0001341439        2022   
8  ORACLE CORP  0001341439        2023   
9  ORACLE CORP  0001341439        2024   

                                                html  
0  https://www.sec.gov/Archives/edgar/data/134143...  
1  https://www.sec.gov/Archives/edgar/data/134143...  
2  https://www.sec.gov/Archives/edgar/data/134143...  
3  https://www.sec.gov/Archives/edgar/data/134143...  
4  https://www.sec.gov/Archives/edgar/data/134143...  
5  https://www.sec.gov/Archives/edgar/data/134143...  
6  https://www.sec.gov/Archives/edgar/